<a href="https://colab.research.google.com/github/rebeccaastaix/Diss/blob/main/Qatarv2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import statsmodels.api as sm

In [2]:
from google.colab import files
uploaded = files.upload()

Saving Qatar_merged_dataset.xlsx to Qatar_merged_dataset.xlsx


In [3]:
file_name = "Qatar_merged_dataset.xlsx"
sheet_name = "Qatar_Model_Data"

df = pd.read_excel(file_name, sheet_name=sheet_name)

# Keep exact columns
df_gas = df[[
    "Year",
    "Non-Oil Real GDP Growth (%)",
    "Qatar Total GDP Growth (annual %)",
    "Oil Price Change",
    "Natural gas rents (% of GDP)",
    "Lag Non-Oil GDP Growth",
    "Lag Total GDP Growth"
]].dropna().copy()

print(df_gas.head())
print(df_gas.tail())
print("Sample years:", df_gas["Year"].min(), "to", df_gas["Year"].max())

   Year  Non-Oil Real GDP Growth (%)  Qatar Total GDP Growth (annual %)  \
1  2001                     8.036937                           3.898187   
2  2002                     5.126883                           7.182152   
3  2003                     5.527286                           3.719959   
4  2004                    19.257439                          19.218915   
5  2005                     9.487953                           7.492758   

   Oil Price Change  Natural gas rents (% of GDP)  Lag Non-Oil GDP Growth  \
1        -14.323012                      7.152238                0.993697   
2          1.623926                      6.070359                8.036937   
3         14.588862                      5.128715                5.126883   
4         32.851885                      5.077344                5.527286   
5         43.025704                      5.443521               19.257439   

   Lag Total GDP Growth  
1              8.028125  
2              3.898187  
3       

In [4]:
# Non-oil GDP explained by gas rents
Y_ng = df_gas["Non-Oil Real GDP Growth (%)"]
X_ng = df_gas[["Natural gas rents (% of GDP)", "Lag Non-Oil GDP Growth"]]
X_ng = sm.add_constant(X_ng)

model_nonoil_gas = sm.OLS(Y_ng, X_ng).fit()
print(model_nonoil_gas.summary())

                                 OLS Regression Results                                
Dep. Variable:     Non-Oil Real GDP Growth (%)   R-squared:                       0.296
Model:                                     OLS   Adj. R-squared:                  0.218
Method:                          Least Squares   F-statistic:                     3.792
Date:                         Sat, 18 Apr 2026   Prob (F-statistic):             0.0422
Time:                                 14:05:33   Log-Likelihood:                -72.193
No. Observations:                           21   AIC:                             150.4
Df Residuals:                               18   BIC:                             153.5
Df Model:                                    2                                         
Covariance Type:                     nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
-----------------------

In [5]:
# Horse race: oil vs gas for non-oil GDP
Y_horse = df_gas["Non-Oil Real GDP Growth (%)"]
X_horse = df_gas[["Oil Price Change", "Natural gas rents (% of GDP)", "Lag Non-Oil GDP Growth"]]
X_horse = sm.add_constant(X_horse)

model_nonoil_oil_gas = sm.OLS(Y_horse, X_horse).fit()
print(model_nonoil_oil_gas.summary())

                                 OLS Regression Results                                
Dep. Variable:     Non-Oil Real GDP Growth (%)   R-squared:                       0.310
Model:                                     OLS   Adj. R-squared:                  0.189
Method:                          Least Squares   F-statistic:                     2.551
Date:                         Sat, 18 Apr 2026   Prob (F-statistic):             0.0899
Time:                                 14:05:57   Log-Likelihood:                -71.983
No. Observations:                           21   AIC:                             152.0
Df Residuals:                               17   BIC:                             156.1
Df Model:                                    3                                         
Covariance Type:                     nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
-----------------------

In [6]:
# Total GDP explained by gas rents
Y_tg = df_gas["Qatar Total GDP Growth (annual %)"]
X_tg = df_gas[["Natural gas rents (% of GDP)", "Lag Total GDP Growth"]]
X_tg = sm.add_constant(X_tg)

model_total_gas = sm.OLS(Y_tg, X_tg).fit()
print(model_total_gas.summary())

                                    OLS Regression Results                                   
Dep. Variable:     Qatar Total GDP Growth (annual %)   R-squared:                       0.394
Model:                                           OLS   Adj. R-squared:                  0.327
Method:                                Least Squares   F-statistic:                     5.849
Date:                               Sat, 18 Apr 2026   Prob (F-statistic):             0.0110
Time:                                       14:06:13   Log-Likelihood:                -67.747
No. Observations:                                 21   AIC:                             141.5
Df Residuals:                                     18   BIC:                             144.6
Df Model:                                          2                                         
Covariance Type:                           nonrobust                                         
                                   coef    std err          

In [7]:
# Horse race: oil vs gas for total GDP
Y_t_horse = df_gas["Qatar Total GDP Growth (annual %)"]
X_t_horse = df_gas[["Oil Price Change", "Natural gas rents (% of GDP)", "Lag Total GDP Growth"]]
X_t_horse = sm.add_constant(X_t_horse)

model_total_oil_gas = sm.OLS(Y_t_horse, X_t_horse).fit()
print(model_total_oil_gas.summary())

                                    OLS Regression Results                                   
Dep. Variable:     Qatar Total GDP Growth (annual %)   R-squared:                       0.486
Model:                                           OLS   Adj. R-squared:                  0.396
Method:                                Least Squares   F-statistic:                     5.368
Date:                               Sat, 18 Apr 2026   Prob (F-statistic):            0.00874
Time:                                       14:06:31   Log-Likelihood:                -66.007
No. Observations:                                 21   AIC:                             140.0
Df Residuals:                                     17   BIC:                             144.2
Df Model:                                          3                                         
Covariance Type:                           nonrobust                                         
                                   coef    std err          

In [10]:
comparison = pd.DataFrame({
    "Model": [
        "Non-oil GDP: gas only",
        "Non-oil GDP: oil + gas",
        "Total GDP: gas only",
        "Total GDP: oil + gas"
    ],
    "R_squared": [
        model_nonoil_gas.rsquared,
        model_nonoil_oil_gas.rsquared,
        model_total_gas.rsquared,
        model_total_oil_gas.rsquared
    ],
    "AIC": [
        model_nonoil_gas.aic,
        model_nonoil_oil_gas.aic,
        model_total_gas.aic,
        model_total_oil_gas.aic
    ]
})

print(comparison)

                    Model  R_squared         AIC
0   Non-oil GDP: gas only   0.296458  150.386280
1  Non-oil GDP: oil + gas   0.310424  151.965224
2     Total GDP: gas only   0.393894  141.493912
3    Total GDP: oil + gas   0.486449  140.014107
